# 14 · Does more background raise domain clustering? (LVIS whole image vs crop; ImageNet)

Notebooks 11–13 asked whether *shared* context (scenes, rooms, activities, co-occurrence) explains BV domain
clustering. Here the question is the *amount* of background in the image, in third-person photos:

- **E · LVIS, same images.** Category means from box crops (object plus a little surround; notebook 12) vs from
  the **whole images** those boxes come from (full scene). Two whole-image versions:
  *all* images containing the category, and *exclusive* images that contain no other category of the set, so
  co-occurring categories do not share identical photos.
- **F · ImageNet.** Whole curated photos with more background than THINGS. Compared with THINGS, BV and VEDB on
  the 75 ImageNet-exclusive categories, with the notebook-12 co-occurrence control.

**Decisions**
- Embeddings: per-image / per-crop L2 normalization, category mean, z-scored within the category set; CLIP
  ViT-B/32 and DINOv3 ViT-B/16 CLS (same models and normalization as every other notebook).
- LVIS crops: the notebook-03 exemplar table (all ground-truth boxes of the category's pooled classes). A check
  cell rebuilds it from per-crop files' metadata to confirm the class pooling used here matches.
- Whole-image LVIS embeddings exist on disk (computed by `data_prep/run_embeddings_imagenet_lvis.sh`); this
  notebook only reads them (one low-priority pass, cached).
- Measures and tests as in notebooks 11–13: AUC (primary), Δ/SD, raw Δ; within-domain category bootstrap.
- BV is 2025.2 (Stage 0), as in notebooks 11–13.

**Outputs:** `outputs/background_amount/results/`, `figures/`; cache in `outputs/background_amount/cache/` (git-ignored).

In [ ]:
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[_v] = '1'   # keep CPU use low (shared machine)
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

# Manuscript figure style (same as analysis/qwen-open-vocab/fig_style.py): Lato, bold axis labels and legends.
import matplotlib.axes, matplotlib.axis, matplotlib.figure
from matplotlib import font_manager
for _f in (Path.cwd().resolve().parents[1] / "talk-figures" / "unr-2026" / "fonts").glob("*.ttf"):
    font_manager.fontManager.addfont(str(_f))
plt.rcParams.update({
    "font.family": ["Lato", "DejaVu Sans", "sans-serif"], "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.labelweight": "bold", "figure.titlesize": 12, "figure.titleweight": "bold",
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#444444", "axes.labelcolor": "#222222",
    "text.color": "#222222", "xtick.color": "#444444", "ytick.color": "#444444"})
def _bold_legend(orig):
    if getattr(orig, "_bold", False):
        return orig
    def legend(self, *args, **kw):
        prop = dict(kw.pop("prop", {}) or {}); prop.setdefault("weight", "bold")
        prop.setdefault("size", kw.pop("fontsize", plt.rcParams["legend.fontsize"]))
        return orig(self, *args, prop=prop, **kw)
    legend._bold = True
    return legend
matplotlib.axes.Axes.legend = _bold_legend(matplotlib.axes.Axes.legend)
matplotlib.figure.Figure.legend = _bold_legend(matplotlib.figure.Figure.legend)
_tick_init = matplotlib.axis.Tick.__init__
if not getattr(_tick_init, "_bold", False):
    def _bold_tick(self, *a, **k):                       # bold tick labels (no rcParam for tick label weight)
        _tick_init(self, *a, **k); self.label1.set_fontweight("bold"); self.label2.set_fontweight("bold")
    _bold_tick._bold = True; matplotlib.axis.Tick.__init__ = _bold_tick
    _grid = matplotlib.axes.Axes.grid                    # no grid lines in any figure
    matplotlib.axes.Axes.grid = lambda self, visible=None, which="major", axis="both", **kw: _grid(self, False, which=which, axis=axis)
plt.rcParams["axes.grid"] = False
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

LVIS_IMAGE_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/image_embeddings")
LVIS_MODEL_DIRS = {"clip": "openai_clip-vit-base-patch32", "dinov3": "facebook_dinov3-vitb16-pretrain-lvd1689m"}
LVIS_SET = EXEMPLAR_SETS / "lvis_exclusive104"
IMAGENET_SET = EXEMPLAR_SETS / "imagenet_exclusive75"
LVIS_CROP_METADATA = MAPPING_OUT / "lvis_crop_metadata.csv.gz"
STAGE0 = MANUSCRIPT_DIR / "exemplar_set_embeddings"
RAW = {
    "bv": str(STAGE0 / "2025.2" / "valid129" / "bv_{m}_exemplar_avg_raw_within_valid129.csv"),
    "things": str(STAGE0 / "valid129" / "things_{m}_exemplar_avg_raw_within_valid129.csv"),
    "vedb": str(VEDB_VALID129 / "bv_{m}_exemplar_avg_raw_within_valid129.csv"),
    "lvis_crop": str(LVIS_SET / "lvis_{m}_exemplar_avg_raw_within_lvis_exclusive104.csv"),
    "imagenet": str(IMAGENET_SET / "imagenet_{m}_exemplar_avg_raw_within_imagenet_exclusive75.csv"),
}
COOC_BV = OUT / "context_controls" / "cache" / "cooccurrence_bv_2025_2.npz"      # notebook 12

BA_OUT = OUT / "background_amount"
BA_CACHE, BA_RESULTS, BA_FIGURES = BA_OUT / "cache", BA_OUT / "results", BA_OUT / "figures"
for d in (BA_CACHE, BA_RESULTS, BA_FIGURES):
    d.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
N_BOOT = 1000
MIN_EXCLUSIVE_IMAGES = 10      # whole-image 'exclusive' version: images with no other category of the set
RECOMPUTE_CACHE = False
LABEL = {"bv": "BabyView", "vedb": "VEDB", "things": "THINGS", "imagenet": "ImageNet",
         "lvis_crop": "LVIS crops", "lvis_whole": "LVIS whole images", "lvis_whole_excl": "LVIS whole images (exclusive)"}
COLOR = {"bv": "#0D47A1", "vedb": "#E07B39", "things": "#90CAF9", "imagenet": "#B9A6E0",
         "lvis_crop": "#4DB38A", "lvis_whole": "#4DB38A", "lvis_whole_excl": "#4DB38A"}    # notebooks 07-09

## Helpers

In [ ]:
def load_cdi_semantic_map() -> dict[str, str]:
    """``uni_lemma`` → CDI semantic domain from ``data/cdi_words.csv`` (same as notebook 09)."""
    df = pd.read_csv(DATA_DIR / "cdi_words.csv", usecols=["uni_lemma", "category"])
    df["uni_lemma"] = df["uni_lemma"].astype(str).str.strip().str.lower()
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.drop_duplicates(subset=["uni_lemma"], keep="first").set_index("uni_lemma")["category"].to_dict()


CDI = load_cdi_semantic_map()
CATS129 = sorted(l.strip().lower() for l in VALID129_TXT.read_text().splitlines() if l.strip())


def load_raw_table(path: str, cats: list[str]) -> np.ndarray:
    df = pd.read_csv(path); df["category"] = df["category"].astype(str).str.strip().str.lower()
    return df.set_index("category").loc[cats].filter(like="dim_").to_numpy(np.float64)


def zscore_rows(X, eps=1e-10):
    return (X - X.mean(axis=0)) / (X.std(axis=0) + eps)


def rdm(X):
    X = X / np.linalg.norm(X, axis=1, keepdims=True); return 1 - X @ X.T


def rank_z(v):
    r = rankdata(v); return (r - r.mean()) / r.std()


def ols(y, X):
    A = np.column_stack([np.ones(len(y))] + X); b, *_ = np.linalg.lstsq(A, y, rcond=None); return b[1:]


def bv_cooccurrence_covariates(cats):
    """Jaccard and shared-context similarity from BV frames (notebook 12 definitions)."""
    z = np.load(COOC_BV); allc = list(z["cats"]); idx = np.array([allc.index(c) for c in cats])
    Cf = z["C"].astype(float); nf = np.diag(Cf).copy(); N = float(z["n_frames"])
    C = Cf[np.ix_(idx, idx)]; n = nf[idx]; jac = C / (n[:, None] + n[None, :] - C + 1e-12)
    ppmi = np.maximum(np.log((Cf * N + 1e-12) / (np.outer(nf, nf) + 1e-12)), 0); np.fill_diagonal(ppmi, 0)
    P = ppmi[idx]; prof = np.zeros((len(idx), len(idx)))
    for a, b in combinations(range(len(idx)), 2):
        keep = np.ones(P.shape[1], bool); keep[[idx[a], idx[b]]] = False; u, v = P[a, keep], P[b, keep]
        prof[a, b] = prof[b, a] = u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12)
    return jac, prof


def measures(R, sem, cov=None, sel=None) -> dict:
    """AUC, Δ, Δ/SD; with ``cov`` also the same-domain coefficient alone / co-occurrence controlled (notebook 12)."""
    sem = np.asarray(sem); sel = np.arange(len(sem)) if sel is None else sel
    s = sem[sel]; iu = np.triu_indices(len(sel), 1); keep = sel[iu[0]] != sel[iu[1]]
    v = R[np.ix_(sel, sel)][iu][keep]; same = (s[iu[0]] == s[iu[1]])[keep]; w, b = v[same], v[~same]
    rk = rankdata(np.concatenate([w, b]))
    out = {"auc": (rk[len(w):].sum() - len(b) * (len(b) + 1) / 2) / (len(w) * len(b)),
           "delta": b.mean() - w.mean(), "delta_sd": (b.mean() - w.mean()) / v.std()}
    if cov is not None:
        jac, prof = cov; y = rank_z(v); sf = same.astype(float); xs = (sf - sf.mean()) / sf.std()
        xj = rank_z(jac[np.ix_(sel, sel)][iu][keep]); xp = rank_z(prof[np.ix_(sel, sel)][iu][keep])
        out["domain_coef_alone"] = -ols(y, [xs])[0]; out["domain_coef_controlled"] = -ols(y, [xs, xj, xp])[0]
    return out


def domain_bootstrap(sem, n, seed):
    sem = np.asarray(sem); rng = np.random.default_rng(seed); doms = sorted(set(sem))
    return [np.concatenate([rng.choice(np.flatnonzero(sem == d), (sem == d).sum()) for d in doms]) for _ in range(n)]


def ci_p(x):
    lo, hi = np.percentile(x, [2.5, 97.5]); return lo, hi, float(2 * min((x <= 0).mean(), (x >= 0).mean()))


def compare(R: dict, sem, pairs, cov=None, seed=3):
    """Per-matrix measures and paired-bootstrap differences (shared draws)."""
    draws = domain_bootstrap(sem, N_BOOT, seed)
    obs = {k: measures(r, sem, cov) for k, r in R.items()}
    bt = {k: pd.DataFrame([measures(r, sem, cov, d) for d in draws]) for k, r in R.items()}
    rows = []
    for a, b in pairs:
        for k in obs[a]:
            lo, hi, p = ci_p(bt[a][k].to_numpy() - bt[b][k].to_numpy())
            rows.append({"pair": f"{a}_minus_{b}", "metric": k, "diff": obs[a][k] - obs[b][k], "ci_lo": lo, "ci_hi": hi, "p_boot": p})
    summ = pd.DataFrame([{"matrix": k, **v, **{f"{m}_ci_lo": np.percentile(bt[k][m], 2.5) for m in v},
                          **{f"{m}_ci_hi": np.percentile(bt[k][m], 97.5) for m in v}} for k, v in obs.items()])
    return summ, pd.DataFrame(rows)

## E · LVIS whole images vs crops

Whole-image file for crop `c`: `{coco_folder}_{coco_image_id:012d}.npy`. A category's images are those with at least
one box of its pooled LVIS classes; *exclusive* images contain no box of any other category in the set.

In [ ]:
LV_CATS = [l.strip().lower() for l in (LVIS_SET / "categories.txt").read_text().splitlines() if l.strip()]
cls = pd.read_csv(LVIS_SET / "lvis_category_classes.csv")
class_to_cdi = dict(zip(cls["lvis_category_id"], cls["cdi_category"].str.strip().str.lower()))
meta = pd.read_csv(LVIS_CROP_METADATA, usecols=["crop_id", "coco_image_id", "category_id", "coco_folder"])
meta = meta[meta["category_id"].isin(class_to_cdi)].copy()
meta["cdi"] = meta["category_id"].map(class_to_cdi)
meta["image_file"] = meta["coco_folder"] + "_" + meta["coco_image_id"].astype(int).map("{:012d}".format) + ".npy"
img_cats = meta.groupby("image_file")["cdi"].agg(lambda s: frozenset(s))
print(f"{len(meta):,} boxes of the {len(LV_CATS)} categories in {len(img_cats):,} images; "
      f"images with ≥2 of the categories: {(img_cats.map(len) > 1).mean():.1%}")
crops_per_cat = meta.groupby("cdi").size()
ref_n = json.loads((LVIS_SET / "lvis_exemplar_embedding_run.json").read_text())["n_exemplars_per_category"]
mism = {c: (int(crops_per_cat.get(c, 0)), ref_n[c]) for c in LV_CATS if int(crops_per_cat.get(c, 0)) != ref_n[c]}
print("crop counts match the notebook-03 exemplar table:", not mism, "" if not mism else list(mism.items())[:5])

In [ ]:
WHOLE_CACHE = BA_CACHE / "lvis_whole_image_category_means.npz"


def build_whole_image_means():
    """One low-priority pass over the per-image files needed; sums per category (all / exclusive images)."""
    files = sorted(img_cats.index); ci = {c: i for i, c in enumerate(LV_CATS)}
    out = {}
    for m, sub in LVIS_MODEL_DIRS.items():
        root = LVIS_IMAGE_EMBED_ROOT / sub; dim = None
        S_all = S_ex = None; n_all = np.zeros(len(LV_CATS)); n_ex = np.zeros(len(LV_CATS)); missing = 0
        for k, f in enumerate(files):
            p = root / f
            if not p.exists():
                missing += 1; continue
            v = np.load(p).astype(np.float64).ravel(); v /= max(np.linalg.norm(v), 1e-12)
            if S_all is None:
                dim = v.size; S_all = np.zeros((len(LV_CATS), dim)); S_ex = np.zeros((len(LV_CATS), dim))
            cs = img_cats[f]
            for c in cs:
                S_all[ci[c]] += v; n_all[ci[c]] += 1
            if len(cs) == 1:
                c = next(iter(cs)); S_ex[ci[c]] += v; n_ex[ci[c]] += 1
            if k % 20000 == 0:
                print(f"  {m}: {k:,}/{len(files):,}", flush=True)
        out.update({f"{m}_sum_all": S_all, f"{m}_n_all": n_all, f"{m}_sum_excl": S_ex, f"{m}_n_excl": n_ex,
                    f"{m}_missing": missing})
    np.savez(WHOLE_CACHE, cats=np.array(LV_CATS), **out)


if RECOMPUTE_CACHE or not WHOLE_CACHE.exists():
    build_whole_image_means()
W = np.load(WHOLE_CACHE); assert list(W["cats"]) == LV_CATS
for m in MODELS:
    print(f"{m}: missing image files {int(W[f'{m}_missing'])}; images per category median "
          f"{np.median(W[f'{m}_n_all']):.0f} (all), {np.median(W[f'{m}_n_excl']):.0f} (exclusive); "
          f"categories with < {MIN_EXCLUSIVE_IMAGES} exclusive images: {int((W[f'{m}_n_excl'] < MIN_EXCLUSIVE_IMAGES).sum())}")

In [ ]:
lv_summ, lv_diff = [], []
for m in MODELS:
    ok = W[f"{m}_n_excl"] >= MIN_EXCLUSIVE_IMAGES
    for setname, keep in (("all_104", np.ones(len(LV_CATS), bool)), ("exclusive_ok", ok)):
        cats = [c for c, k in zip(LV_CATS, keep) if k]; ix = np.flatnonzero(keep); sem = [CDI.get(c, "other") for c in cats]
        R = {"lvis_crop": rdm(zscore_rows(load_raw_table(RAW["lvis_crop"].format(m=m), cats))),
             "lvis_whole": rdm(zscore_rows(W[f"{m}_sum_all"][ix] / W[f"{m}_n_all"][ix, None])),
             "bv": rdm(zscore_rows(load_raw_table(RAW["bv"].format(m=m), cats))),
             "things": rdm(zscore_rows(load_raw_table(RAW["things"].format(m=m), cats)))}
        pairs = [("lvis_whole", "lvis_crop"), ("bv", "lvis_whole"), ("things", "lvis_whole")]
        if setname == "exclusive_ok":
            R["lvis_whole_excl"] = rdm(zscore_rows(W[f"{m}_sum_excl"][ix] / W[f"{m}_n_excl"][ix, None]))
            pairs += [("lvis_whole_excl", "lvis_crop"), ("lvis_whole_excl", "lvis_whole")]
        s, d = compare(R, sem, pairs)
        lv_summ.append(s.assign(model=m, category_set=setname, n_categories=len(cats)))
        lv_diff.append(d.assign(model=m, category_set=setname, n_categories=len(cats)))
lv_summ = pd.concat(lv_summ, ignore_index=True); lv_diff = pd.concat(lv_diff, ignore_index=True)
lv_summ.to_csv(BA_RESULTS / "lvis_whole_vs_crop_summary.csv", index=False)
lv_diff.to_csv(BA_RESULTS / "lvis_whole_vs_crop_differences.csv", index=False)
display(lv_summ.pivot_table(index=["model", "category_set", "n_categories"], columns="matrix", values="auc").round(3))
display(lv_diff[lv_diff.metric == "auc"].round(3))

## F · ImageNet (whole curated photos) vs THINGS, BV, VEDB — 75 categories

Same measures plus the notebook-12 same-domain coefficient with BV co-occurrence controls.

In [ ]:
IN_CATS = [l.strip().lower() for l in (IMAGENET_SET / "categories.txt").read_text().splitlines() if l.strip()]
vedb_cats = set(pd.read_csv(RAW["vedb"].format(m="clip"), usecols=["category"])["category"].str.strip().str.lower())
assert set(IN_CATS) <= set(CATS129) and set(IN_CATS) <= vedb_cats
sem_in = [CDI.get(c, "other") for c in IN_CATS]; cov_in = bv_cooccurrence_covariates(IN_CATS)
print(len(IN_CATS), "categories;", pd.Series(sem_in).value_counts().to_dict())
in_summ, in_diff = [], []
for m in MODELS:
    R = {ds: rdm(zscore_rows(load_raw_table(RAW[ds].format(m=m), IN_CATS))) for ds in ("bv", "vedb", "things", "imagenet")}
    s, d = compare(R, sem_in, [("imagenet", "things"), ("bv", "imagenet"), ("vedb", "imagenet"), ("bv", "things"), ("bv", "vedb")], cov=cov_in)
    in_summ.append(s.assign(model=m)); in_diff.append(d.assign(model=m))
in_summ = pd.concat(in_summ, ignore_index=True); in_diff = pd.concat(in_diff, ignore_index=True)
in_summ.to_csv(BA_RESULTS / "imagenet_comparison_summary.csv", index=False)
in_diff.to_csv(BA_RESULTS / "imagenet_comparison_differences.csv", index=False)
display(in_summ[["model", "matrix", "auc", "delta", "delta_sd", "domain_coef_alone", "domain_coef_controlled"]].round(3))
display(in_diff.pivot_table(index=["model", "pair"], columns="metric", values=["diff", "p_boot"]).round(3))

## Figure

In [ ]:
def fig_background():
    """AUC with 95% CI: LVIS crop vs whole vs whole-exclusive (exclusive-ok set), and ImageNet vs others (75)."""
    fig, axes = plt.subplots(2, len(MODELS), figsize=(12.5, 6.2), sharex="col", layout="constrained")
    LD = pd.read_csv(BA_OUT / "results" / "lvis_whole_vs_crop_differences.csv"); LD = LD[LD.category_set == "exclusive_ok"]
    ID = pd.read_csv(BA_OUT / "results" / "imagenet_comparison_differences.csv")
    CMP = [{"lvis_whole": ("lvis_whole_minus_lvis_crop", "vs crop"), "lvis_whole_excl": ("lvis_whole_excl_minus_lvis_crop", "vs crop"),
            "bv": ("bv_minus_lvis_whole", "vs whole"), "things": ("things_minus_lvis_whole", "vs whole")},
           {"bv": ("bv_minus_imagenet", "vs ImageNet"), "vedb": ("vedb_minus_imagenet", "vs ImageNet"),
            "things": ("imagenet_minus_things", "vs ImageNet")}]                      # AUC differences, category bootstrap p
    star = lambda p: "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "n.s."

    for col, m in enumerate(MODELS):
        for row, (tab, keys, title) in enumerate((
                (lv_summ[(lv_summ.model == m) & (lv_summ.category_set == "exclusive_ok")],
                 ["bv", "things", "lvis_crop", "lvis_whole", "lvis_whole_excl"], "LVIS crop vs whole"),
                (in_summ[in_summ.model == m], ["bv", "vedb", "things", "imagenet"], "ImageNet"))):
            ax = axes[row, col]; t = tab.set_index("matrix")
            for y, k in enumerate(keys):
                ax.plot([t.loc[k, "auc_ci_lo"], t.loc[k, "auc_ci_hi"]], [y, y], color="0.35", lw=1.5, zorder=1)
                hollow = k == "lvis_crop"
                ax.scatter([t.loc[k, "auc"]], [y], s=65, zorder=3, edgecolor="0.2", lw=1.0,
                           facecolor="white" if hollow else COLOR[k], marker="s" if k == "lvis_whole_excl" else "o")
                D_ = LD if row == 0 else ID; cmp = CMP[row].get(k)
                p_ = D_[(D_.model == m) & (D_.metric == "auc") & (D_.pair == cmp[0])].p_boot if cmp else []
                sig_txt = f"   {cmp[1]}: {star(p_.iloc[0])}" if len(p_) else ""
                ax.annotate(f"{t.loc[k, 'auc']:.3f}{sig_txt}", (t.loc[k, "auc_ci_hi"], y), xytext=(5, 0),
                            textcoords="offset points", va="center", fontsize=8.5, color="0.25")
            n = int(tab.n_categories.iloc[0]) if "n_categories" in tab else len(IN_CATS)
            ax.set_yticks(range(len(keys)), [LABEL[k] for k in keys]); ax.set_ylim(len(keys) - 0.5, -0.5)
            ax.set_title(f"{'CLIP' if m == 'clip' else 'DINOv3'} · {title} ({n})", fontsize=10)
            ax.grid(axis="x", color="0.9"); ax.set_axisbelow(True)
            for sp in ("top", "right"):
                ax.spines[sp].set_visible(False)
            ax.set_xlabel("AUC: P(between > within distance)", fontsize=9)
    out = BA_FIGURES / "background_amount_auc.png"; fig.savefig(out, dpi=200); fig.savefig(out.with_suffix(".pdf")); plt.close(fig)
    return out


show_images([fig_background()], width=820)

### Review checklist

- Do LVIS crop counts match the notebook-03 exemplar table (same class pooling)? Any missing whole-image files?
- Whole vs crop: does adding the full scene raise or lower clustering? Does the *exclusive* version (no shared
  photos between categories) agree?
- ImageNet vs THINGS: does more background in curated photos raise clustering?
- ImageNet pools many classes per word (dog = 118 breeds), which adds within-category variety.